# Gemma4Good Field Report - Self-contained Kaggle Notebook

This notebook recreates the Gemma4Good Field Report workflow for Kaggle without the live website, live API, camera, GPS hardware, or separately hosted model service.

Workflow: choose or upload an image through Kaggle input, collect field metadata, download the Gemma 4 GGUF model file into Kaggle working storage when enabled, run image-first triage, review or correct the draft, then save the reviewed report as JSON and CSV.

The notebook does not package multi-GB weights inside the `.ipynb`. Instead it downloads the GGUF from Hugging Face at runtime. The deterministic triage engine remains as a transparent fallback so the notebook still runs from top to bottom if the download is disabled, interrupted, or unavailable.


## 1. Runtime Setup

Run all cells in order. The notebook writes outputs to `/kaggle/working` when running on Kaggle, or to the local working directory elsewhere.

In [ ]:
from __future__ import annotations

import base64
import csv
import io
import hashlib
import json
import math
import os
import re
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    from IPython.display import HTML, display
except Exception:
    class HTML(str):
        pass
    def display(value: Any) -> None:
        print(value)
from PIL import Image, ImageStat

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT_DIR = KAGGLE_WORKING / 'gemma4good-fieldreport-output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SUPPORTED_CATEGORIES = ['water_quality', 'sanitation', 'infrastructure', 'health_risk']
SUPPORTED_SEVERITIES = ['low', 'medium', 'high', 'critical']

print(f'Output directory: {OUTPUT_DIR}')


## 2. User Inputs

For Kaggle, add an image dataset or upload an image into the notebook session, then set `USER_IMAGE_PATH` to that file path. If this stays blank, the notebook will use the first image found under `/kaggle/input`. If no image exists, it generates a demo image so the workflow still runs end to end.

Edit the metadata values before running all cells if you want a specific report.

In [ ]:
# Optional: set this to an image path such as '/kaggle/input/my-dataset/photo.jpg'.
USER_IMAGE_PATH = ''

# Model download configuration. This avoids packaging a 16GB model file in the notebook.
DOWNLOAD_GGUF_MODEL = True
GGUF_MODEL_URL = 'https://huggingface.co/ggml-org/gemma-4-26B-A4B-it-GGUF/resolve/main/gemma-4-26B-A4B-it-Q4_K_M.gguf?download=true'
GGUF_MODEL_FILENAME = 'gemma-4-26B-A4B-it-Q4_K_M.gguf'
GGUF_MODEL_DIR = str(OUTPUT_DIR / 'model-files')

# Set this to a small integer only for connectivity testing. Leave as None for the full model.
# Example: DOWNLOAD_TEST_BYTES = 1048576 downloads only the first 1 MiB.
DOWNLOAD_TEST_BYTES = None

# Field metadata. Blank values are allowed; the notebook will keep the workflow runnable.
FIELD_METADATA = {
    'captured_at': '',  # ISO timestamp. Blank means now in UTC.
    'city': 'Mogadishu',
    'district': 'Wadajir',
    'latitude': '',
    'longitude': '',
    'field_note': 'Standing water near a hand pump with visible discoloration and waste nearby.',
}

# Optional corrections after the draft triage. Leave empty to accept the draft.
# You can override: category, severity, summary, observed_issues, recommended_actions,
# needs_escalation, confidence, note.
REVIEW_OVERRIDES = {
    # 'severity': 'high',
    # 'note': 'Reviewed by field team; priority repair requested.',
}

TRIAGE_MODE = 'gguf_download_then_deterministic_fallback'
print(f'Triage mode: {TRIAGE_MODE}')
print('GGUF download enabled:', DOWNLOAD_GGUF_MODEL)


## 3. Image Selection and Demo Fallback

This cell resolves the report image. It uses a user-provided file first, then searches Kaggle input data, then uses the bundled `01a-captured-image.jpg` field image. The bundled image is also embedded in the notebook so this demo path does not generate a synthetic image.


In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
BUNDLED_DEMO_IMAGE_NAME = '01a-captured-image.jpg'
BUNDLED_DEMO_IMAGE_B64 = """..."""


def find_first_input_image() -> Path | None:
    if not KAGGLE_INPUT.exists():
        return None
    preferred = sorted(KAGGLE_INPUT.rglob(BUNDLED_DEMO_IMAGE_NAME))
    for path in preferred:
        if path.is_file():
            return path
    for path in sorted(KAGGLE_INPUT.rglob('*')):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            return path
    return None


def find_bundled_demo_image() -> Path | None:
    candidates = [
        Path.cwd() / BUNDLED_DEMO_IMAGE_NAME,
        Path.cwd() / 'submission-notebook' / BUNDLED_DEMO_IMAGE_NAME,
        Path('submission-notebook') / BUNDLED_DEMO_IMAGE_NAME,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def write_embedded_demo_image() -> Path:
    demo_path = OUTPUT_DIR / BUNDLED_DEMO_IMAGE_NAME
    if not demo_path.exists():
        demo_path.write_bytes(base64.b64decode(BUNDLED_DEMO_IMAGE_B64))
    return demo_path


def resolve_image_path(user_path: str) -> tuple[Path, str]:
    if user_path.strip():
        candidate = Path(user_path.strip())
        if candidate.exists():
            return candidate, 'user_path'
        print(f'User image path not found, falling back: {candidate}')

    bundled = find_bundled_demo_image()
    if bundled:
        return bundled, 'bundled_demo_01a'

    discovered = find_first_input_image()
    if discovered:
        source = 'kaggle_input_01a' if discovered.name == BUNDLED_DEMO_IMAGE_NAME else 'kaggle_input'
        return discovered, source

    return write_embedded_demo_image(), 'embedded_demo_01a'


image_path, image_source = resolve_image_path(USER_IMAGE_PATH)
image = Image.open(image_path).convert('RGB')
print(f'Image source: {image_source}')
print(f'Image path: {image_path}')
print(f'Image size: {image.width} x {image.height}')

if plt:
    plt.figure(figsize=(7, 4.5))
    plt.imshow(image)
    plt.axis('off')
    plt.title('Selected field image')
    plt.show()
else:
    display(image.resize((min(640, image.width), int(min(640, image.width) * image.height / image.width))))


## 4. Download the Gemma 4 GGUF Model

This cell downloads the Gemma 4 GGUF file from Hugging Face into Kaggle working storage. It does not use the deployed Gemma4Good API or any private host.

The full file is large, so the download function supports resume. For connectivity testing only, set `DOWNLOAD_TEST_BYTES` to a small value such as `1048576`; the notebook will issue a range request and stop after that many bytes.


In [ ]:
@dataclass
class ModelBundle:
    downloaded: bool
    source: str = ''
    local_path: str = ''
    bytes_downloaded: int = 0
    complete: bool = False
    error: str = ''


def human_bytes(value: int | float) -> str:
    value = float(value or 0)
    for unit in ['B', 'KiB', 'MiB', 'GiB', 'TiB']:
        if value < 1024 or unit == 'TiB':
            return f'{value:.1f} {unit}'
        value /= 1024


def download_url_to_file(url: str, destination: Path, test_bytes: int | None = None) -> ModelBundle:
    import urllib.error
    import urllib.request

    destination.parent.mkdir(parents=True, exist_ok=True)
    partial_destination = destination.with_suffix(destination.suffix + '.part') if test_bytes else destination
    existing = partial_destination.stat().st_size if partial_destination.exists() else 0

    headers = {}
    if test_bytes:
        end_byte = max(0, int(test_bytes) - 1)
        headers['Range'] = f'bytes=0-{end_byte}'
        mode = 'wb'
        existing = 0
    elif existing > 0:
        headers['Range'] = f'bytes={existing}-'
        mode = 'ab'
    else:
        mode = 'wb'

    request = urllib.request.Request(url, headers=headers)
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            status = getattr(response, 'status', None)
            total_header = response.headers.get('Content-Length')
            total_remaining = int(total_header) if total_header and total_header.isdigit() else None
            expected_total = existing + total_remaining if total_remaining is not None and not test_bytes else test_bytes
            print(f'Download response status: {status}')
            if expected_total:
                print(f'Expected bytes this artifact: {human_bytes(expected_total)}')

            chunk_size = 1024 * 1024
            written = existing
            with partial_destination.open(mode + '') as output:
                while True:
                    chunk = response.read(chunk_size)
                    if not chunk:
                        break
                    output.write(chunk)
                    written += len(chunk)
                    if written == len(chunk) or written % (256 * 1024 * 1024) < chunk_size:
                        print(f'Downloaded {human_bytes(written)}')
                    if test_bytes and written >= test_bytes:
                        break

        if test_bytes:
            return ModelBundle(
                downloaded=True,
                source=url,
                local_path=str(partial_destination),
                bytes_downloaded=partial_destination.stat().st_size,
                complete=False,
                error=f'test download stopped after {human_bytes(partial_destination.stat().st_size)}',
            )

        return ModelBundle(
            downloaded=True,
            source=url,
            local_path=str(destination),
            bytes_downloaded=destination.stat().st_size,
            complete=True,
        )
    except Exception as exc:
        size = partial_destination.stat().st_size if partial_destination.exists() else 0
        return ModelBundle(
            downloaded=False,
            source=url,
            local_path=str(partial_destination),
            bytes_downloaded=size,
            complete=False,
            error=f'{type(exc).__name__}: {exc}',
        )


def maybe_download_gguf_model() -> ModelBundle:
    if not DOWNLOAD_GGUF_MODEL:
        return ModelBundle(downloaded=False, source=GGUF_MODEL_URL, error='DOWNLOAD_GGUF_MODEL is False')

    destination = Path(GGUF_MODEL_DIR) / GGUF_MODEL_FILENAME
    if DOWNLOAD_TEST_BYTES is None and destination.exists() and destination.stat().st_size > 1024 * 1024:
        print(f'Using existing GGUF file: {destination} ({human_bytes(destination.stat().st_size)})')
        return ModelBundle(
            downloaded=True,
            source=GGUF_MODEL_URL,
            local_path=str(destination),
            bytes_downloaded=destination.stat().st_size,
            complete=True,
        )

    return download_url_to_file(GGUF_MODEL_URL, destination, DOWNLOAD_TEST_BYTES)


MODEL_BUNDLE = maybe_download_gguf_model()
if MODEL_BUNDLE.downloaded:
    print(f'GGUF available at: {MODEL_BUNDLE.local_path}')
    print(f'Bytes downloaded: {MODEL_BUNDLE.bytes_downloaded} ({human_bytes(MODEL_BUNDLE.bytes_downloaded)})')
    print(f'Complete model file: {MODEL_BUNDLE.complete}')
else:
    print('GGUF model was not downloaded.')
    print(f'Reason: {MODEL_BUNDLE.error}')


## 5. Image-first Triage Engine

The live app uses Gemma4 for image-first triage. This notebook now downloads the GGUF artifact from Hugging Face when enabled, but it does not start a llama.cpp runtime inside the notebook. The deterministic local triage engine remains the executable notebook path and records whether the GGUF download completed.


In [ ]:
CATEGORY_KEYWORDS = {
    'water_quality': ['dirty', 'turbid', 'brown', 'green', 'algae', 'smell', 'odor', 'oil', 'discolor', 'cloudy', 'contaminated'],
    'sanitation': ['trash', 'waste', 'sewage', 'latrine', 'open defecation', 'garbage', 'fecal', 'dump', 'drain'],
    'infrastructure': ['pump', 'pipe', 'broken', 'leak', 'borehole', 'well', 'tap', 'tank', 'repair', 'valve', 'generator'],
    'health_risk': ['illness', 'diarrhea', 'cholera', 'mosquito', 'stagnant', 'standing water', 'clinic', 'children', 'risk', 'outbreak'],
}

SEVERITY_KEYWORDS = {
    'critical': ['cholera', 'outbreak', 'severe', 'urgent', 'hospital', 'critical', 'no water'],
    'high': ['leak', 'sewage', 'stagnant', 'standing water', 'broken', 'contaminated', 'smell', 'illness'],
    'medium': ['dirty', 'trash', 'cloudy', 'repair', 'waste', 'discoloration'],
    'low': ['minor', 'monitor', 'routine'],
}

CATEGORY_LABELS = {
    'water_quality': 'Water quality',
    'sanitation': 'Sanitation',
    'infrastructure': 'Infrastructure',
    'health_risk': 'Health risk',
}

TRIAGE_JSON_SCHEMA_HINT = {
    'category': 'one of: water_quality, sanitation, infrastructure, health_risk',
    'severity': 'one of: low, medium, high, critical',
    'summary': 'concise field triage summary',
    'observed_issues': ['short issue 1', 'short issue 2'],
    'recommended_actions': ['short action 1', 'short action 2'],
    'needs_escalation': True,
    'confidence': 0.0,
}


def normalize_timestamp(value: str) -> str:
    value = str(value or '').strip()
    if value:
        try:
            return datetime.fromisoformat(value.replace('Z', '+00:00')).astimezone(timezone.utc).isoformat()
        except Exception:
            pass
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


def image_features(img: Image.Image) -> dict[str, float]:
    small = img.resize((96, 96))
    stat = ImageStat.Stat(small)
    mean_r, mean_g, mean_b = stat.mean[:3]
    brightness = (mean_r + mean_g + mean_b) / 3.0
    green_bias = mean_g - ((mean_r + mean_b) / 2.0)
    brown_bias = mean_r - mean_b
    contrast = sum(stat.stddev[:3]) / 3.0
    digest = hashlib.sha256(img.tobytes()[:200000]).hexdigest()
    hash_score = int(digest[:8], 16) / 0xFFFFFFFF
    return {
        'mean_r': mean_r,
        'mean_g': mean_g,
        'mean_b': mean_b,
        'brightness': brightness,
        'green_bias': green_bias,
        'brown_bias': brown_bias,
        'contrast': contrast,
        'hash_score': hash_score,
    }


def score_categories(note: str, features: dict[str, float]) -> dict[str, float]:
    text = note.lower()
    scores = {category: 0.0 for category in SUPPORTED_CATEGORIES}
    for category, words in CATEGORY_KEYWORDS.items():
        for word in words:
            if word in text:
                scores[category] += 2.0

    if features['green_bias'] > 8 or features['brown_bias'] > 18:
        scores['water_quality'] += 1.2
    if features['contrast'] > 48:
        scores['sanitation'] += 0.9
        scores['infrastructure'] += 0.6
    if features['brightness'] < 95:
        scores['health_risk'] += 0.7
    if features['hash_score'] > 0.66:
        scores['infrastructure'] += 0.4
    elif features['hash_score'] < 0.33:
        scores['sanitation'] += 0.4

    if max(scores.values()) == 0:
        scores['water_quality'] = 1.0
    return scores


def choose_severity(note: str, category_scores: dict[str, float], features: dict[str, float]) -> str:
    text = note.lower()
    severity_score = 1.0
    for severity, words in SEVERITY_KEYWORDS.items():
        hits = sum(1 for word in words if word in text)
        if hits:
            severity_score = max(severity_score, SUPPORTED_SEVERITIES.index(severity) + 1 + min(hits, 2) * 0.25)
    if category_scores.get('health_risk', 0) >= 2.0:
        severity_score += 0.5
    if category_scores.get('sanitation', 0) >= 2.0 and category_scores.get('water_quality', 0) >= 2.0:
        severity_score += 0.6
    if features['contrast'] > 62 or features['brightness'] < 75:
        severity_score += 0.4

    if severity_score >= 3.7:
        return 'critical'
    if severity_score >= 2.7:
        return 'high'
    if severity_score >= 1.7:
        return 'medium'
    return 'low'


def issue_templates(category: str, severity: str) -> tuple[list[str], list[str]]:
    observed = {
        'water_quality': ['Possible discoloration or contamination in the water source.', 'Image and note suggest water-quality review is needed.'],
        'sanitation': ['Waste or drainage conditions may affect the water point.', 'Sanitation conditions should be checked around the source.'],
        'infrastructure': ['Water access hardware may need inspection or repair.', 'Pump, pipe, tank, or tap condition should be verified.'],
        'health_risk': ['Conditions may create elevated public-health exposure.', 'Escalation should consider nearby households and vulnerable users.'],
    }[category]
    actions = {
        'water_quality': ['Collect a water-quality sample if field kits are available.', 'Inspect source protection and notify water committee.'],
        'sanitation': ['Clear immediate waste where safe and document drainage path.', 'Coordinate sanitation follow-up with local operators.'],
        'infrastructure': ['Inspect pump, pipe, valve, and storage components.', 'Create a repair ticket with image, location, and severity.'],
        'health_risk': ['Notify the health or WASH focal point for risk review.', 'Prioritize mitigation if households rely on this source.'],
    }[category]
    if severity in {'high', 'critical'}:
        actions.insert(0, 'Escalate for same-day review if this is an active community source.')
    return observed, actions


def normalize_triage(raw: dict[str, Any], source: str, fallback_reason: str = '') -> dict[str, Any]:
    category = str(raw.get('category') or '').strip().lower()
    if category not in SUPPORTED_CATEGORIES:
        category = 'water_quality'
    severity = str(raw.get('severity') or '').strip().lower()
    if severity not in SUPPORTED_SEVERITIES:
        severity = 'medium'

    observed = raw.get('observed_issues', raw.get('observedIssues', []))
    actions = raw.get('recommended_actions', raw.get('recommendedActions', []))
    if not isinstance(observed, list):
        observed = [str(observed)] if observed else []
    if not isinstance(actions, list):
        actions = [str(actions)] if actions else []
    default_observed, default_actions = issue_templates(category, severity)

    confidence = raw.get('confidence', 0.5)
    try:
        confidence = max(0.0, min(1.0, float(confidence)))
    except Exception:
        confidence = 0.5

    needs_escalation = raw.get('needs_escalation', raw.get('needsEscalation', severity in {'high', 'critical'}))
    if isinstance(needs_escalation, str):
        needs_escalation = needs_escalation.strip().lower() in {'true', 'yes', '1'}

    return {
        'triage_mode': source,
        'category': category,
        'severity': severity,
        'summary': str(raw.get('summary') or f'{CATEGORY_LABELS[category]} concern; review image and metadata before dispatch.')[:500],
        'observed_issues': [str(item)[:180] for item in (observed or default_observed)[:6]],
        'recommended_actions': [str(item)[:180] for item in (actions or default_actions)[:6]],
        'needs_escalation': bool(needs_escalation),
        'confidence': round(confidence, 2),
        'fallback_reason': fallback_reason,
    }


def triage_with_fallback(img: Image.Image, metadata: dict[str, Any], reason: str = '') -> dict[str, Any]:
    note = str(metadata.get('field_note') or '')
    features = image_features(img)
    category_scores = score_categories(note, features)
    category = max(category_scores, key=category_scores.get)
    severity = choose_severity(note, category_scores, features)
    observed, actions = issue_templates(category, severity)
    top_score = category_scores[category]
    total_score = sum(category_scores.values()) or 1.0
    confidence = max(0.42, min(0.89, 0.45 + (top_score / total_score) * 0.34 + (0.08 if note else 0.0)))
    needs_escalation = severity in {'high', 'critical'} or category == 'health_risk'

    location = ', '.join(part for part in [metadata.get('district'), metadata.get('city')] if part) or 'manual location not provided'
    summary = f"{CATEGORY_LABELS[category]} concern at {location}; severity is {severity}. Review image and field metadata before dispatch."

    result = normalize_triage({
        'category': category,
        'severity': severity,
        'summary': summary,
        'observed_issues': observed,
        'recommended_actions': actions,
        'needs_escalation': needs_escalation,
        'confidence': confidence,
    }, source='deterministic_fallback', fallback_reason=reason)
    result['category_scores'] = {k: round(v, 2) for k, v in category_scores.items()}
    result['image_features'] = {k: round(v, 2) for k, v in features.items()}
    return result



## 6. Create the Draft Report

This mirrors the live app's image-first order: image plus location/time first, model triage when available, then review.


In [ ]:
def report_id_for(image_path: Path, metadata: dict[str, Any]) -> str:
    seed = f"{image_path.name}|{metadata.get('captured_at')}|{metadata.get('city')}|{metadata.get('district')}"
    return 'g4g-' + hashlib.sha256(seed.encode('utf-8')).hexdigest()[:12]

metadata = dict(FIELD_METADATA)
metadata['captured_at'] = normalize_timestamp(metadata.get('captured_at', ''))
metadata['captured_at_source'] = 'manual_or_notebook'
metadata['city'] = str(metadata.get('city') or '').strip()
metadata['district'] = str(metadata.get('district') or '').strip()
metadata['field_note'] = str(metadata.get('field_note') or '').strip()
metadata['latitude'] = str(metadata.get('latitude') or '').strip()
metadata['longitude'] = str(metadata.get('longitude') or '').strip()

draft = triage_with_fallback(image, metadata, reason=MODEL_BUNDLE.error if not MODEL_BUNDLE.complete else 'GGUF downloaded; deterministic triage used because notebook does not launch llama.cpp runtime')

draft_report = {
    'report_id': report_id_for(image_path, metadata),
    'status': 'draft',
    'created_at': datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    'image': {
        'path': str(image_path),
        'source': image_source,
        'width': image.width,
        'height': image.height,
    },
    'metadata': metadata,
    'analysis': draft,
    'model_runtime': {
        'requested': DOWNLOAD_GGUF_MODEL,
        'downloaded': MODEL_BUNDLE.downloaded,
        'complete': MODEL_BUNDLE.complete,
        'source': MODEL_BUNDLE.source,
        'local_path': MODEL_BUNDLE.local_path,
        'bytes_downloaded': MODEL_BUNDLE.bytes_downloaded,
        'error': MODEL_BUNDLE.error,
    },
}

fallback_note = ''
if draft.get('triage_mode') != 'local_model':
    fallback_note = f"<p><b>Fallback reason:</b> {draft.get('fallback_reason') or 'n/a'}</p>"

display(HTML(f"""
<div style="font-family:Arial,sans-serif;border:1px solid #d0d7de;border-radius:8px;padding:14px;max-width:820px">
  <div style="font-size:12px;color:#57606a;text-transform:uppercase;letter-spacing:.08em">Gemma4Good draft triage</div>
  <h2 style="margin:.25rem 0">{draft['category']} - {draft['severity']}</h2>
  <p>{draft['summary']}</p>
  <p><b>Escalation:</b> {'yes' if draft['needs_escalation'] else 'no'} | <b>Confidence:</b> {draft['confidence']}</p>
  <p><b>Mode:</b> {draft['triage_mode']}</p>
  <p><b>GGUF downloaded:</b> {'yes' if MODEL_BUNDLE.downloaded else 'no'} | <b>Complete:</b> {'yes' if MODEL_BUNDLE.complete else 'no'}</p>
  {fallback_note}
</div>
"""))

print(json.dumps(draft_report, indent=2))



## 7. Review and Correct

The live app asks the field user to confirm or correct the AI draft before submit. In this notebook, edit `REVIEW_OVERRIDES` in the parameter cell and rerun from here, or leave it empty to accept the draft. List fields can be provided as Python lists or semicolon-separated strings.


In [ ]:
def coerce_list(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    return [part.strip() for part in str(value).split(';') if part.strip()]

def apply_review_overrides(report: dict[str, Any], overrides: dict[str, Any]) -> dict[str, Any]:
    reviewed = json.loads(json.dumps(report))
    analysis = reviewed['analysis']
    metadata = reviewed['metadata']

    if overrides.get('category') in SUPPORTED_CATEGORIES:
        analysis['category'] = overrides['category']
    if overrides.get('severity') in SUPPORTED_SEVERITIES:
        analysis['severity'] = overrides['severity']
    if 'summary' in overrides and str(overrides['summary']).strip():
        analysis['summary'] = str(overrides['summary']).strip()
    if 'observed_issues' in overrides:
        items = coerce_list(overrides['observed_issues'])
        if items:
            analysis['observed_issues'] = items
    if 'recommended_actions' in overrides:
        items = coerce_list(overrides['recommended_actions'])
        if items:
            analysis['recommended_actions'] = items
    if 'needs_escalation' in overrides:
        analysis['needs_escalation'] = bool(overrides['needs_escalation'])
    if 'confidence' in overrides:
        try:
            analysis['confidence'] = round(float(overrides['confidence']), 2)
        except Exception:
            pass
    if 'note' in overrides and str(overrides['note']).strip():
        metadata['field_note'] = str(overrides['note']).strip()

    reviewed['status'] = 'reviewed'
    reviewed['review'] = {
        'reviewed_at': datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        'overrides_applied': sorted(overrides.keys()),
    }
    return reviewed

reviewed_report = apply_review_overrides(draft_report, REVIEW_OVERRIDES)
print(json.dumps(reviewed_report, indent=2))

## 8. Submit and Export

Submitting in this notebook means writing the reviewed report to local notebook output files. This replaces the deployed `/api/fieldreport/submit` call with Kaggle-local files.


In [ ]:
def flatten_report(report: dict[str, Any]) -> dict[str, Any]:
    analysis = report['analysis']
    metadata = report['metadata']
    image_info = report['image']
    runtime = report.get('model_runtime', {})
    return {
        'report_id': report['report_id'],
        'status': report['status'],
        'created_at': report['created_at'],
        'reviewed_at': report.get('review', {}).get('reviewed_at', ''),
        'triage_mode': analysis.get('triage_mode', ''),
        'model_requested': runtime.get('requested', ''),
        'model_downloaded': runtime.get('downloaded', ''),
        'model_complete': runtime.get('complete', ''),
        'model_source': runtime.get('source', ''),
        'model_local_path': runtime.get('local_path', ''),
        'model_bytes_downloaded': runtime.get('bytes_downloaded', ''),
        'model_error': runtime.get('error', ''),
        'category': analysis.get('category', ''),
        'severity': analysis.get('severity', ''),
        'summary': analysis.get('summary', ''),
        'observed_issues': ' | '.join(analysis.get('observed_issues', [])),
        'recommended_actions': ' | '.join(analysis.get('recommended_actions', [])),
        'needs_escalation': analysis.get('needs_escalation', False),
        'confidence': analysis.get('confidence', ''),
        'captured_at': metadata.get('captured_at', ''),
        'city': metadata.get('city', ''),
        'district': metadata.get('district', ''),
        'latitude': metadata.get('latitude', ''),
        'longitude': metadata.get('longitude', ''),
        'field_note': metadata.get('field_note', ''),
        'image_path': image_info.get('path', ''),
        'image_source': image_info.get('source', ''),
    }

json_path = OUTPUT_DIR / 'gemma4good_reviewed_report.json'
jsonl_path = OUTPUT_DIR / 'gemma4good_reviewed_reports.jsonl'
csv_path = OUTPUT_DIR / 'gemma4good_reviewed_report.csv'

with json_path.open('w', encoding='utf-8') as f:
    json.dump(reviewed_report, f, indent=2)

with jsonl_path.open('a', encoding='utf-8') as f:
    f.write(json.dumps(reviewed_report) + '\n')

flat = flatten_report(reviewed_report)
with csv_path.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(flat.keys()))
    writer.writeheader()
    writer.writerow(flat)

print('Submitted reviewed report locally.')
print(f'JSON: {json_path}')
print(f'JSONL append log: {jsonl_path}')
print(f'CSV: {csv_path}')

if pd:
    display(pd.DataFrame([flat]))
else:
    print(flat)



## 9. Final Report Summary

This final cell gives a compact review of the saved field report.


In [ ]:
analysis = reviewed_report['analysis']
metadata = reviewed_report['metadata']

display(HTML(f"""
<div style="font-family:Arial,sans-serif;border:1px solid #0969da;border-radius:8px;padding:16px;max-width:900px;background:#f6f8fa">
  <div style="font-size:12px;color:#57606a;text-transform:uppercase;letter-spacing:.08em">Reviewed Gemma4Good field report</div>
  <h2 style="margin:.3rem 0">{reviewed_report['report_id']}</h2>
  <p><b>Category:</b> {analysis['category']} | <b>Severity:</b> {analysis['severity']} | <b>Escalate:</b> {'yes' if analysis['needs_escalation'] else 'no'} | <b>Confidence:</b> {analysis['confidence']}</p>
  <p><b>Location:</b> {metadata.get('district') or 'n/a'}, {metadata.get('city') or 'n/a'} | <b>Captured:</b> {metadata.get('captured_at')}</p>
  <p>{analysis['summary']}</p>
  <p><b>Mode:</b> {analysis.get('triage_mode')}</p>
</div>
"""))

print('Observed issues:')
for item in analysis.get('observed_issues', []):
    print(f'- {item}')

print('\nRecommended actions:')
for item in analysis.get('recommended_actions', []):
    print(f'- {item}')